# The Great Haversine Expedition of 2026

AKA: How a ~30 line C++ implementation took nearly **5 hours** to debug.

## Issue #1 — MSVC was not being used
### Symptom:

```shell
cl is not recognized
```

or

```shell
C:/Strawberry/c/bin/c++.exe
```

Appearing in build logs.


### Cause:

CMake was discovering MinGW/Strawberry GCC instead of VS.

### Fix:

Open:

```shell
x64 Native Tools Command Prompt for VS
```

and configure CMake from there.

## Issue #2 — Wrong Architecture

### Symptom:

```shell
Python is 64-bit, chosen compiler is 32-bit
```

### Cause:

VS shell was using:

```shell
Hostx86/x86
```

However, Python was 64-bit.

### Fix:

Use:
```shell
Hostx64/x64
```
toolchain

## Issue #3 — Pybind11 not found

### Symptom:

```shell
Could not find pybind11Config.cmake
```

### Cause:

CMake could not discover pybind11's CMake files.

### Fix:

```shell
python -m pybind11 --cmakedir
```

and pass:

```shell
-Dpybind11_DIR=<that path>
```

to CMake

## Issue #4 — Windows CMD Quoting

### Symptom:

```shell
Ignoring extra path from command line
```

### Cause:

Path contained spaces, i.e:

```shell
C:\Users\Tahir C D\...
```

and CMake received malformed arguments.

### Fix:

Quote the entire argument:

```shell
"-Dpybind11_DIR=..."
```

## Issue #5 — Missing MSVC runtime headers

### Symptom:

```shell
corecrt_math_defines.h not found
```

or

```shell
vcruntime.h not found
```

### Cause:

Incorrect compiler environment.

### Fix:

Use the proper VS x64 developer shell.

Also remove:

```cpp
#include <corecrt_math_defines.h>
```

and define:

```cpp
constexpr double PI = 3.14159265358979323846;
```

## Issue #6 — Haversine Not Exposed to Python

### Symptom:

```py
dir(fast_module)
```

Showed:

```py
['add', 'parallel_sum', 'power']
```

but no:

```py
'fast_haversine'
```

### Cause:

Function existed in C++, but was not bound to pybind11.

### Fix:

```cpp
m.def("fast_haversine", &geo::fast_haversine);
```

## Issue #7 — Missing Namespace Brace

### Symptom:

```shell
expected '}' at the end of input
```

### Cause:

Deleted `int main()` from fast_module.cpp and accidentally removed the ending brace of the outer namespace definition directly preceeding it

### Fix:

Replace the missing brace.

## Issue #8 — OpenMP flags were being passed incorrectly

### Symptom:

```shell
LINK: warning LNK4044:
unrecognized option '/openmp'
```

### Cause:
Previous CMakeLists.txt was manually injecting OpenMP flags:

```cmake
target_compile_options(...)
target_link_options(...)
```

### Fix:

Use:

```cmake
target_link_libraries(
    fast_module
    PRIVATE
    OpenMP::OpenMP_CXX
)
```

And let CMake take the wheel.

## Issue #9 — Debug Runtime DLL Hell

### Symptom:

```shell
ImportError:
DLL load failed while importing fast_module
```

### Cause:

The built module depended on:

```shell
MSVCP140D.dll
VCOMP140D.dll
VCRUNTIME140D.dll
ucrtbased.dll
```

Files with the `D` suffix were debug runtime

### Fix:

Build `Release` instead of `Debug`

# Lessons Learned

**C++ was not the hard part.**

While the actual Haversine implementation took only ~30 lines, the toolchain debugging had me deal with:

- MSVC
- CMake
- Ninja
- pybind11
- OpenMP
- Python ABI
- DLL dependencies
- Windows Shell behavior

**THE COMPILER WAS USUALLY RIGHT.**

Almost every useful clue came directly from error messages, once interpreted correctly.

**DUMPBIN IS YOUR FRIEND.**

...Also maybe just use a Linux dev environment next time.

In [17]:
from fast_module import fast_haversine

In [15]:
points = (1.3521, 103.8189, 1.3644, 103.9915)
result = fast_haversine(*points)
if int(result) == 19:
    print("😭 This took 5 hours")

😭 This took 5 hours
